# Lock v4 confirmation: the partner-selection test (opens the confirmation pool once)

Reads `locked_config_v4.json` and `locked_config_v4_amendment1.json`, checks every hash, then reads the confirmation pool **once**.

- **Predictors** (all from the estimation pool, file `04_results/pair_quantities_estimation_v4.csv`, unchanged): C, rho_cost, partner marginal cost.
- **Primary outcome** (amendment v4.1): anchor-relative PDR rescued under the union rule, each model at its own 25% eye-level threshold, averaged over the 9 seed combinations.
- **Primary statistic**: mean over 11 anchors of the within-anchor Spearman between estimation C and the confirmation outcome; architecture bootstrap (1,000); supported if positive and the 95% interval excludes zero.
- **Also reported** as locked: partial correlation controlling for partner marginal quality; anchor permutation interval; decision-relevant secondary (least overlap versus best marginal, per-anchor paired difference); rule comparison with random and oracle; incremental referrals and incremental false positives per pair; the matched-volume separation named in the lock's limitation; everything with and without retfound_adversarial.

CPU is enough. About 5 to 10 minutes.

In [1]:
# Cell 1: guards and locked parameters
from google.colab import drive
drive.mount('/content/drive')
import os, json, hashlib, itertools, datetime
import numpy as np, pandas as pd
from scipy.stats import rankdata

BASE    = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS, AUDIT = (f'{BASE}/{d}' for d in ('01_splits','03_predictions','04_results','99_audit'))
SPLIT_FILE = f'{SPLITS}/eyepacs_split_kaggletest_estconf_v2.csv'
LOCK_FILE  = f'{RESULTS}/locked_config_v4.json'
AMEND_FILE = f'{RESULTS}/locked_config_v4_amendment1.json'
PAIRQ_FILE = f'{RESULTS}/pair_quantities_estimation_v4.csv'
OUT_FILE   = f'{RESULTS}/confirmation_results_v4.json'

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''): h.update(blk)
    return h.hexdigest()

assert not os.path.exists(OUT_FILE), 'Confirmation already opened once. Do not rerun.'
LOCK  = json.load(open(LOCK_FILE)); AMEND = json.load(open(AMEND_FILE))
LOCK_SHA = sha256_file(LOCK_FILE)
assert AMEND['amends_sha256'] == LOCK_SHA, 'Amendment is not chained to this lock file.'
assert AMEND['outcome_definition'] in ('anchor_relative', 'pair_min')
OUTCOME = AMEND['outcome_definition']

BURDEN = LOCK['definitions']['burden']; assert LOCK['definitions']['referral_unit'] == 'eye'
N_BOOT, N_PERM, RNG_SEED = 1000, 10000, 20260923
ROSTER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
SEED_TAGS = ['', '_seed1', '_seed2']
print(f'Lock v4 {LOCK_SHA[:16]}..., amendment written {AMEND["written_at_utc"]}; outcome = {OUTCOME}')

Mounted at /content/drive
Lock v4 9b78257ece485d02..., amendment written 2026-09-23T13:04:28.343103+00:00; outcome = anchor_relative


In [2]:
# Cell 2: estimation-pool predictors (frozen before this notebook)
pq = pd.read_csv(PAIRQ_FILE)
PQ_SHA = sha256_file(PAIRQ_FILE)
assert os.path.getmtime(PAIRQ_FILE) < os.path.getmtime(AMEND_FILE), 'Predictor file changed after the amendment.'
EST = {}
for _, r in pq.iterrows():
    for a, b in ((r.model_a, r.model_b), (r.model_b, r.model_a)):
        EST[(a, b)] = {'C': r.C, 'rho_cost': r.rho_cost, 'partner_marginal': r.marginal_b if a == r.model_a else r.marginal_a}
assert len(EST) == 110
print('Estimation predictors loaded for 110 ordered pairs.')

Estimation predictors loaded for 110 ordered pairs.


In [3]:
# Cell 3: open the confirmation pool (the one read)
split = pd.read_csv(SPLIT_FILE)
assert sha256_file(SPLIT_FILE) == LOCK['file_sha256'][os.path.basename(SPLIT_FILE)]
def pick(df, c, w):
    for x in c:
        if x in df.columns: return x
    raise AssertionError(f'{w} column not found: {list(df.columns)}')
ID  = pick(split, ['image_id','image','id','filename'], 'image id')
PAT = pick(split, ['patient_id','patient','pid','subject_id'], 'patient id')
GR  = pick(split, ['level','grade','true_grade','diagnosis'], 'grade')
POOL= pick(split, ['pool','split','set','assignment','estconf'], 'pool')
lc = split[POOL].astype(str).str.lower()
CONF_KEY = next(v for v in set(lc) if v.startswith('conf'))
conf = split[lc == CONF_KEY].sort_values(ID).reset_index(drop=True)
IDS = conf[ID].astype(str).to_numpy(); grade = conf[GR].to_numpy().astype(int)
n = len(IDS); referable = grade >= 2; pdr = grade == 4
print(f'Confirmation pool: {n:,} eyes, {conf[PAT].nunique():,} patients, {pdr.sum()} PDR, referable {referable.mean():.4f}')

P = {}
for a in ROSTER:
    for t in SEED_TAGS:
        fn = f'eyepacs_predictions_kaggletest_{a}{t}_v2.csv'; path = f'{PREDS}/{fn}'
        assert sha256_file(path) == LOCK['file_sha256'][fn], f'{fn} changed since lock.'
        df = pd.read_csv(path)
        idc = pick(df, ['image_id','image','id','filename'], 'image id')
        pc = pick(df, ['predicted_probability','prob_referable','probability','prob','p_referable'], 'probability')
        df[idc] = df[idc].astype(str)
        df = df.drop_duplicates(subset=idc).set_index(idc).reindex(IDS)
        assert df[pc].notna().all(), f'{fn}: confirmation eyes missing.'
        P[f'{a}{t}'] = df[pc].to_numpy(dtype=np.float64)
OPENED_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat()
print('Opened at', OPENED_UTC)

Confirmation pool: 17,858 eyes, 8,929 patients, 382 PDR, referable 0.1941
Opened at 2026-09-23T14:11:14.422952+00:00


In [4]:
# Cell 4: confirmation quantities for every ordered pair, averaged over 9 seed combinations
def refer(s, frac):
    k = int(round(frac * len(s))); m = np.zeros(len(s), bool); m[np.argsort(-s, kind='stable')[:k]] = True; return m
def pct_rank(s): return rankdata(s) / len(s)

REF = {k: refer(v, BURDEN) for k, v in P.items()}
RNK = {k: pct_rank(v) for k, v in P.items()}

def ordered(a, b):
    acc = {k: [] for k in ('rescued','pair_min','incr_ref','incr_fp','mv_net')}
    for ta in SEED_TAGS:
        ra, xa = REF[f'{a}{ta}'], RNK[f'{a}{ta}']
        for tb in SEED_TAGS:
            rb, xb = REF[f'{b}{tb}'], RNK[f'{b}{tb}']
            miss_a, miss_b, both = pdr & ~ra, pdr & ~rb, pdr & ~ra & ~rb
            acc['rescued'].append(miss_a.sum() - both.sum())
            acc['pair_min'].append(min(miss_a.sum(), miss_b.sum()) - both.sum())
            acc['incr_ref'].append((rb & ~ra).sum())
            acc['incr_fp'].append((rb & ~ra & ~referable).sum())
            ens = refer(np.maximum(xa, xb), BURDEN)          # matched volume, max of percentile ranks
            acc['mv_net'].append((pdr & ~ra).sum() - (pdr & ~ens).sum())
    return {k: float(np.mean(v)) for k, v in acc.items()}

CONFQ = {(a, b): ordered(a, b) for a in ROSTER for b in ROSTER if a != b}
OUTK = 'rescued' if OUTCOME == 'anchor_relative' else 'pair_min'
print(f'Mean anchor-relative PDR rescued per ordered pair: {np.mean([v["rescued"] for v in CONFQ.values()]):.2f}')
print(f'Mean incremental referrals {np.mean([v["incr_ref"] for v in CONFQ.values()]):.0f}, of which false positives {np.mean([v["incr_fp"] for v in CONFQ.values()]):.0f}')

Mean anchor-relative PDR rescued per ordered pair: 12.75
Mean incremental referrals 1377, of which false positives 1075


In [5]:
# Cell 5: primary, partial, permutation, architecture bootstrap
def spear(x, y, z=None):
    x, y = rankdata(x), rankdata(y)
    if z is not None:
        Z = np.column_stack([np.ones(len(z)), rankdata(z)])
        x = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]; y = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    return float(np.corrcoef(x, y)[0, 1]) if np.std(x) > 0 and np.std(y) > 0 else np.nan

def stat(anchors_multiset, pred='C', partial=False, roster=ROSTER):
    vals = []
    for a in anchors_multiset:
        cands = [c for c in anchors_multiset if c != a and c in roster]
        if len(set(cands)) < 3: continue
        x = [EST[(a, c)][pred] * (-1 if pred == 'rho_cost' else 1) for c in cands]
        y = [CONFQ[(a, c)][OUTK] for c in cands]
        z = [-EST[(a, c)]['partner_marginal'] for c in cands] if partial else None
        r = spear(x, y, z)
        if np.isfinite(r): vals.append(r)
    return float(np.mean(vals)) if vals else np.nan

def boot(fn, roster):
    rng = np.random.default_rng(RNG_SEED); out = []
    for _ in range(N_BOOT):
        ms = list(rng.choice(roster, size=len(roster), replace=True)); out.append(fn(ms))
    out = np.array(out); return [float(np.nanpercentile(out, 2.5)), float(np.nanpercentile(out, 97.5))]

def perm_interval(roster, pred='C'):
    rng = np.random.default_rng(RNG_SEED + 1); obs = []
    for _ in range(N_PERM):
        vals = []
        for a in roster:
            cands = [c for c in roster if c != a]
            x = rng.permutation([EST[(a, c)][pred] for c in cands]); y = [CONFQ[(a, c)][OUTK] for c in cands]
            vals.append(spear(x, y))
        obs.append(np.nanmean(vals))
    return [float(np.percentile(obs, 2.5)), float(np.percentile(obs, 97.5))]

def analysis(roster):
    res = {}
    for label, pred, partial in (('C', 'C', False), ('C_partial_on_partner_marginal', 'C', True),
                                 ('rho_cost_negated', 'rho_cost', False), ('partner_quality', 'partner_marginal', False)):
        f = (lambda ms, p=pred, q=partial: stat(ms, p, q, roster)) if pred != 'partner_marginal' else None
        if pred == 'partner_marginal':
            def f(ms):
                vals = []
                for a in ms:
                    cands = [c for c in ms if c != a]
                    if len(set(cands)) < 3: continue
                    r = spear([-EST[(a, c)]['partner_marginal'] for c in cands], [CONFQ[(a, c)][OUTK] for c in cands])
                    if np.isfinite(r): vals.append(r)
                return float(np.mean(vals)) if vals else np.nan
        est = f(list(roster)); ci = boot(f, roster)
        res[label] = {'estimate': round(est, 4), 'arch_bootstrap_95': [round(v, 4) for v in ci]}
    res['C']['permutation_null_95'] = [round(v, 4) for v in perm_interval(roster)]
    res['C']['supported'] = bool(res['C']['estimate'] > 0 and res['C']['arch_bootstrap_95'][0] > 0)
    return res

R_ALL = analysis(ROSTER)
R_NOADV = analysis([a for a in ROSTER if a != 'retfound_adversarial'])
print(json.dumps(R_ALL, indent=1))

{
 "C": {
  "estimate": 0.3368,
  "arch_bootstrap_95": [
   -0.1255,
   0.6473
  ],
  "permutation_null_95": [
   -0.2015,
   0.1952
  ],
  "supported": false
 },
 "C_partial_on_partner_marginal": {
  "estimate": 0.443,
  "arch_bootstrap_95": [
   -0.0826,
   0.7129
  ]
 },
 "rho_cost_negated": {
  "estimate": 0.2819,
  "arch_bootstrap_95": [
   -0.1994,
   0.6637
  ]
 },
 "partner_quality": {
  "estimate": 0.4967,
  "arch_bootstrap_95": [
   -0.0889,
   0.8624
  ]
 }
}


In [6]:
# Cell 6: rule comparison, decision-relevant secondary, matched-volume separation
def rules(roster):
    per = {}
    for a in roster:
        cands = [c for c in roster if c != a]
        pick = {'least_ceiling_overlap': max(cands, key=lambda c: EST[(a, c)]['C']),
                'least_weighted_dependence': min(cands, key=lambda c: EST[(a, c)]['rho_cost']),
                'best_partner_marginal': min(cands, key=lambda c: EST[(a, c)]['partner_marginal'])}
        out = {r: CONFQ[(a, c)][OUTK] for r, c in pick.items()}
        out['random'] = float(np.mean([CONFQ[(a, c)][OUTK] for c in cands]))
        out['oracle'] = float(max(CONFQ[(a, c)][OUTK] for c in cands))
        out['picks'] = pick
        out['mv_net_least_overlap'] = CONFQ[(a, pick['least_ceiling_overlap'])]['mv_net']
        out['mv_net_best_marginal'] = CONFQ[(a, pick['best_partner_marginal'])]['mv_net']
        out['incr_fp_least_overlap'] = CONFQ[(a, pick['least_ceiling_overlap'])]['incr_fp']
        out['incr_fp_best_marginal'] = CONFQ[(a, pick['best_partner_marginal'])]['incr_fp']
        per[a] = out
    rn = ['least_ceiling_overlap','least_weighted_dependence','best_partner_marginal','random','oracle']
    means = {r: round(float(np.mean([per[a][r] for a in roster])), 3) for r in rn}
    gain = {r: round((means[r] - means['random']) / (means['oracle'] - means['random']), 3) if means['oracle'] > means['random'] else None
            for r in rn[:3]}
    def diff_boot(key1, key2):
        rng = np.random.default_rng(RNG_SEED + 2); d = np.array([per[a][key1] - per[a][key2] for a in roster]); bs = []
        for _ in range(N_BOOT): bs.append(rng.choice(d, size=len(d), replace=True).mean())
        return {'mean': round(float(d.mean()), 3), 'arch_bootstrap_95': [round(float(np.percentile(bs, 2.5)), 3), round(float(np.percentile(bs, 97.5)), 3)],
                'anchors_favouring_first': int((d > 0).sum()), 'ties': int((d == 0).sum())}
    return {'per_anchor': per, 'rule_means': means, 'fraction_of_oracle_gain': gain,
            'decision_secondary_least_overlap_minus_best_marginal': diff_boot('least_ceiling_overlap', 'best_partner_marginal'),
            'matched_volume_least_overlap_minus_best_marginal': diff_boot('mv_net_least_overlap', 'mv_net_best_marginal'),
            'incr_fp_least_overlap_minus_best_marginal': diff_boot('incr_fp_least_overlap', 'incr_fp_best_marginal')}

RULES_ALL = rules(ROSTER); RULES_NOADV = rules([a for a in ROSTER if a != 'retfound_adversarial'])
for k in ('rule_means','fraction_of_oracle_gain','decision_secondary_least_overlap_minus_best_marginal',
          'matched_volume_least_overlap_minus_best_marginal','incr_fp_least_overlap_minus_best_marginal'):
    print(k, RULES_ALL[k])

rule_means {'least_ceiling_overlap': 12.616, 'least_weighted_dependence': 12.202, 'best_partner_marginal': 13.848, 'random': 12.754, 'oracle': 16.03}
fraction_of_oracle_gain {'least_ceiling_overlap': -0.042, 'least_weighted_dependence': -0.168, 'best_partner_marginal': 0.334}
decision_secondary_least_overlap_minus_best_marginal {'mean': -1.232, 'arch_bootstrap_95': [-3.404, 0.939], 'anchors_favouring_first': 4, 'ties': 2}
matched_volume_least_overlap_minus_best_marginal {'mean': -4.606, 'arch_bootstrap_95': [-7.323, -1.838], 'anchors_favouring_first': 1, 'ties': 1}
incr_fp_least_overlap_minus_best_marginal {'mean': 203.152, 'arch_bootstrap_95': [33.43, 384.066], 'anchors_favouring_first': 10, 'ties': 1}


In [7]:
# Cell 7: write results (once) and log
pairs_tbl = [{'anchor': a, 'partner': b, 'est_C': round(EST[(a, b)]['C'], 5), 'est_rho_cost': round(EST[(a, b)]['rho_cost'], 5),
              'est_partner_marginal': round(EST[(a, b)]['partner_marginal'], 2), **{f'conf_{k}': round(v, 3) for k, v in CONFQ[(a, b)].items()}}
             for (a, b) in CONFQ]
pd.DataFrame(pairs_tbl).to_csv(f'{RESULTS}/confirmation_pairs_v4.csv', index=False)
out = {'lock_sha256': LOCK_SHA, 'amendment_written_utc': AMEND['written_at_utc'], 'predictor_file_sha256': PQ_SHA,
       'confirmation_opened_utc': OPENED_UTC, 'outcome_definition': OUTCOME,
       'pool': {'eyes': int(n), 'patients': int(conf[PAT].nunique()), 'pdr_eyes': int(pdr.sum())},
       'primary_all_11': R_ALL, 'primary_without_retfound_adversarial': R_NOADV,
       'rules_all_11': {k: v for k, v in RULES_ALL.items() if k != 'per_anchor'},
       'rules_without_retfound_adversarial': {k: v for k, v in RULES_NOADV.items() if k != 'per_anchor'},
       'per_anchor_all_11': RULES_ALL['per_anchor'],
       'pre_registered_expectation': AMEND['pseudo_confirmation_preview_estimation_only'],
       'definitions': {'incremental_referrals': 'eyes referred by the partner and not by the anchor, each at its own 25% threshold',
                       'incremental_false_positives': 'incremental referrals with grade 0 or 1',
                       'matched_volume_net': 'anchor PDR misses minus PDR misses of max(percentile rank) ensemble referring 25%'}}
with open(OUT_FILE, 'w') as f: json.dump(out, f, indent=2)
with open(f'{AUDIT}/RUN_LOG.md', 'a') as f:
    f.write(f'''\n\n## Lock v4 confirmation ({OPENED_UTC[:10]})\n\nConfirmation pool opened once at {OPENED_UTC}. Primary (within-anchor Spearman, C vs {OUTCOME} PDR): '''
            f'''{R_ALL["C"]["estimate"]} (architecture bootstrap {R_ALL["C"]["arch_bootstrap_95"]}); supported = {R_ALL["C"]["supported"]}. '''
            f'''Outputs 04_results/confirmation_results_v4.json, confirmation_pairs_v4.csv. Produced by 99_audit/CONFIRM_lock_v4.ipynb.\n''')
print(json.dumps({k: out[k] for k in ('primary_all_11','primary_without_retfound_adversarial')}, indent=1))

{
 "primary_all_11": {
  "C": {
   "estimate": 0.3368,
   "arch_bootstrap_95": [
    -0.1255,
    0.6473
   ],
   "permutation_null_95": [
    -0.2015,
    0.1952
   ],
   "supported": false
  },
  "C_partial_on_partner_marginal": {
   "estimate": 0.443,
   "arch_bootstrap_95": [
    -0.0826,
    0.7129
   ]
  },
  "rho_cost_negated": {
   "estimate": 0.2819,
   "arch_bootstrap_95": [
    -0.1994,
    0.6637
   ]
  },
  "partner_quality": {
   "estimate": 0.4967,
   "arch_bootstrap_95": [
    -0.0889,
    0.8624
   ]
  }
 },
 "primary_without_retfound_adversarial": {
  "C": {
   "estimate": 0.2814,
   "arch_bootstrap_95": [
    -0.216,
    0.5893
   ],
   "permutation_null_95": [
    -0.2196,
    0.221
   ],
   "supported": false
  },
  "C_partial_on_partner_marginal": {
   "estimate": 0.3886,
   "arch_bootstrap_95": [
    -0.2167,
    0.6329
   ]
  },
  "rho_cost_negated": {
   "estimate": 0.2134,
   "arch_bootstrap_95": [
    -0.2651,
    0.6522
   ]
  },
  "partner_quality": {
   "e

## After this notebook
Send back `04_results/confirmation_results_v4.json`. Move this executed notebook to `99_audit/CONFIRM_lock_v4.ipynb`. It refuses to run a second time.